# Merge clinical_outcome DEG results into `bulk_counts.h5ad`

Loads `bulk_counts.h5ad`, attaches the pydeseq2 clinical_outcome differential expression results (`DEG_results.csv`) onto `adata.var`, and writes a new h5ad (`lzf`-compressed) without touching the original file.

In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import scanpy as sc
import pandas as pd

In [ ]:
H5AD_IN = f"{ACC_DATA_ROOT}/bulk_rna_data/h5ad_files/bulk_counts.h5ad"
DEG_CSV = f"{ACC_DATA_ROOT}/bulk_rna_data/outputs/pydeseq2_out/clinical_outcome/DEG_results.csv"
H5AD_OUT = f"{ACC_DATA_ROOT}/bulk_rna_data/h5ad_files/clinical_outcome/bulk_counts_clinical_outcome.h5ad"

In [ ]:
adata = sc.read_h5ad(H5AD_IN)
deg = pd.read_csv(DEG_CSV, index_col="Geneid")

print(adata)
deg.head()

In [ ]:
# adata.var.index is just a positional RangeIndex, not gene IDs -- the real
# Ensembl gene ID lives in adata.var['gene_id']. Align the DEG table to it
# explicitly rather than assuming row order matches.
deg_aligned = deg.reindex(adata.var["gene_id"])

n_missing = deg_aligned.isna().all(axis=1).sum()
assert n_missing == 0, f"{n_missing} gene_id(s) in adata.var have no matching row in DEG_results.csv"
assert len(deg_aligned) == adata.n_vars

In [ ]:
# Overwrites any stale DE columns already present in adata.var from a previous merge.
for col in deg_aligned.columns:
    adata.var[col] = deg_aligned[col].values

adata.var.head()

In [ ]:
Path(H5AD_OUT).parent.mkdir(parents=True, exist_ok=True)
adata.write_h5ad(H5AD_OUT, compression="lzf")

print(adata)
print("Saved to", H5AD_OUT)